<a href="https://colab.research.google.com/github/jech2nNexus/awesome-openclaw-usecases/blob/main/_ai_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "gymnasium[mujoco]"
import gymnasium

env = gymnasium.make("Pusher-v5")
episodes = 50

for episode in range(episodes):
    observation, info = env.reset()
    done = False
    truncated = False

    while not (done or truncated):
        # Exploration: 무작위 행동 선택
        action = env.action_space.sample()
        observation, reward, done, truncated, info = env.step(action)

env.close()
print(f"{episodes}회의 에피소드 탐색이 완료되었습니다.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 22.8 MB/s eta 0:00:00
50회의 에피소드 탐색이 완료되었습니다.


In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import glob
import io
import base64
from IPython.display import HTML, display

# 시각화를 위해 render_mode를 'rgb_array'로 설정
env = gym.make("Pusher-v5", render_mode="rgb_array")

# 비디오 녹화를 위한 Wrapper 추가 (1개의 에피소드만 녹화)
env = RecordVideo(env, video_folder='./video', episode_trigger=lambda e: True)

observation, info = env.reset()
done = False
truncated = False

# 1 에피소드 동안 무작위 행동 수행
while not (done or truncated):
    action = env.action_space.sample()
    observation, reward, done, truncated, info = env.step(action)

env.close()

# 녹화된 비디오 파일을 Colab에서 재생
video_files = glob.glob('./video/*.mp4')
if len(video_files) > 0:
    # 가장 최근에 생성된 비디오 선택
    video_path = video_files[-1]
    video = io.open(video_path, 'r+b').read()
    encoded = base64.b64encode(video)
    display(HTML(data='''
        <video width="640" height="480" controls autoplay loop>
            <source src="data:video/mp4;base64,{0}" type="video/mp4" />
        </video>'''.format(encoded.decode('ascii'))))
else:
    print("비디오를 찾을 수 없습니다.")

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


In [ ]:
!pip install stable-baselines3[extra]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 19.0 MB/s eta 0:00:00


In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

# 환경 생성
env = gym.make("Pusher-v5")

# PPO 모델 정의 (MlpPolicy 사용)
model = PPO("MlpPolicy", env, verbose=1)

# 모델 학습 (빠른 데모를 위해 100,000 타임스텝 지정. 필요시 늘릴 수 있습니다)
print("PPO 모델 학습을 시작합니다...")
model.learn(total_timesteps=1)

# 학습된 모델 저장
model.save("ppo_pusher")
print("모델이 'ppo_pusher.zip'로 저장되었습니다.")
env.close()

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import PPO
import glob
import io
import base64
from IPython.display import HTML, display

# 시각화를 위한 환경 및 레코더 설정
env = gym.make("Pusher-v5", render_mode="rgb_array")
env = RecordVideo(env, video_folder='./video_ppo', episode_trigger=lambda e: True)

# 저장된 PPO 모델 로드
model = PPO.load("ppo_pusher")

observation, info = env.reset()
done = False
truncated = False

# 학습된 모델을 사용하여 1 에피소드 진행
while not (done or truncated):
    # deterministic=True로 설정하여 확률적이 아닌 결정론적(최적) 행동 선택
    action, _states = model.predict(observation, deterministic=True)
    observation, reward, done, truncated, info = env.step(action)

env.close()

# 녹화된 비디오 파일을 Colab에서 재생
video_files = glob.glob('./video_ppo/*.mp4')
if len(video_files) > 0:
    # 가장 최근에 생성된 비디오 선택
    video_path = max(video_files, key=os.path.getctime)
    video = io.open(video_path, 'r+b').read()
    encoded = base64.b64encode(video)
    display(HTML(data='''
        <video width="640" height="480" controls autoplay loop>
            <source src="data:video/mp4;base64,{0}" type="video/mp4" />
        </video>'''.format(encoded.decode('ascii'))))
else:
    print("비디오를 찾을 수 없습니다.")